In [1]:
from sklearn.manifold import TSNE
from io import StringIO
import pandas as pd
import numpy as np
from src import taxonomy
import plotly.express as px
import matplotlib.colors as mcolors


def parse_dist_mat_iqtree(dist_data: str) -> pd.DataFrame:
    """
    Parse distance matrices generated by iqtree.
    dist_data: str
    Contents of distance matrix file as string.
    """
    dist_str = dist_data.replace(' \n', '\n')
    dist_str = dist_str.split("\n",1)[1]
    dist_buf = StringIO(dist_str)
    df = pd.read_csv(dist_buf, header=None, index_col=0, sep=r'\s+')
    df.columns = df.index
    return df

def load_dist_mat_iqtree(path: str) -> pd.DataFrame:
    with open(path, 'r') as ml_dist:
        mat = parse_dist_mat_iqtree(ml_dist.read())
    return mat

In [19]:
def plot(mds_mat: np.ndarray, tax_df: pd.DataFrame, color_column: str, colormap_path: str = 'colormap-oom-order.csv', output: str = 'tsne'):
    colormap_df = pd.read_csv(colormap_path)
    color_discrete_map = dict(zip(colormap_df['order'], colormap_df['color']))

    display_col = color_column.title()
    df_plot = pd.DataFrame({
        'Accession': tax_df.index,
        display_col: tax_df[color_column]
            .str.replace(r'^[a-z]__', '', regex=True)
            .replace('', 'Unknown')
            .fillna('Unknown'),
        'Species': tax_df['species'],
        't-SNE 1': mds_mat[:, 0],
        't-SNE 2': mds_mat[:, 1]
    })

    fig = px.scatter(
        df_plot,
        x='t-SNE 1',
        y='t-SNE 2',
        color=display_col,
        color_discrete_map=color_discrete_map,
        hover_name='Accession',
        hover_data={'Species': True, 't-SNE 1': False, 't-SNE 2': False},
        title=None,
        width=800,
        height=600,
        render_mode='svg',
    )

    fig.update_traces(marker=dict(size=10, line=dict(width=1, color='black')))
    fig.update_layout(
        font=dict(color='black'),
        legend=dict(itemsizing='trace'),
        margin=dict(b=120),
        paper_bgcolor='white',
        plot_bgcolor='white',
        xaxis=dict(
            showgrid=True,
            gridcolor='lightgrey',
            zeroline=False,
            showline=True,
            linecolor='black',
            mirror=True,
            ticks='outside',
            tickcolor='black',
        ),
        yaxis=dict(
            showgrid=True,
            gridcolor='lightgrey',
            zeroline=False,
            showline=True,
            linecolor='black',
            mirror=True,
            ticks='outside',
            tickcolor='black',
        ),
    )

    fig.write_html(f'{output}.html')
    fig.write_image(f'{output}.svg')
    fig.show()

In [3]:
iqtree_dist = load_dist_mat_iqtree('data/cox2_th_final/distances/oom-msa.fasta.mldist')
iqtree_dist

,LC595970.1,LC902567.1,LC902566.1,PQ771703.1,MW928685.1,OQ970163.1,MW820927.1,MW820923.1,ON532704.1,MH325651.1,...,GU361009.1,GU361007.1,GU360992.1,KT257437.1,EU826096.1,KY569077.1,MN006990.1,PV764605.1,AB437405.1,KM210528.1
0,,,,,,,,,,,,,,,,,,,,,
LC595970.1,0.000000,0.436644,0.383225,0.318466,0.346672,0.346672,0.348113,0.355360,0.333308,0.303388,...,0.233228,0.213200,0.224422,0.128248,0.078784,1.154192,0.503663,0.769321,0.724297,1.098043
LC902567.1,0.436644,0.000000,0.033552,0.283321,0.457298,0.457298,0.459013,0.467627,0.473642,0.517912,...,0.710613,0.676095,0.727533,0.542056,0.505597,1.862189,0.943365,1.095013,0.904420,1.035002
LC902566.1,0.383225,0.033552,0.000000,0.302626,0.454200,0.454200,0.444086,0.464470,0.458336,0.535694,...,0.561196,0.550615,0.593320,0.435949,0.393039,1.594038,0.886578,0.949780,0.782127,1.000439
PQ771703.1,0.318466,0.283321,0.302626,0.000000,0.458375,0.458375,0.448132,0.456652,0.439722,0.473402,...,0.558943,0.530590,0.522010,0.414921,0.425095,1.144251,0.629644,0.882251,0.929803,1.198657
MW928685.1,0.346672,0.457298,0.454200,0.458375,0.000000,0.000001,0.003523,0.003509,0.018435,0.048504,...,0.629123,0.642355,0.673661,0.410742,0.391302,1.476975,0.657320,0.766445,0.783085,0.719898
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
KY569077.1,1.154192,1.862189,1.594038,1.144251,1.476975,1.476975,1.445246,1.471862,1.453191,1.345429,...,1.375537,1.639570,1.468865,1.501343,1.379285,0.000000,1.018734,1.032604,0.978562,1.494028
MN006990.1,0.503663,0.943365,0.886578,0.629644,0.657320,0.657320,0.659952,0.657320,0.599699,0.556696,...,0.747493,0.696678,0.750616,0.490646,0.483500,1.018734,0.000000,0.193084,0.602140,0.914666
PV764605.1,0.769321,1.095013,0.949780,0.882251,0.766445,0.766445,0.747110,0.766445,0.691817,0.751527,...,0.880465,0.950050,0.875034,0.774442,0.538961,1.032604,0.193084,0.000000,0.671534,0.986183


In [ ]:
# IQ-TREE excludes identical sequences from the distance matrix but re-adds them to the tree.
# Restore the 12 missing sequences by copying rows/columns from their identical counterparts.
identical_seqs = {
    'KJ654242.1': 'DQ365729.1',
    'KJ654236.1': 'DQ365729.1',
    'MG552674.1': 'DQ365729.1',
    'KP330598.1': 'MT079815.1',
    'KF269675.1': 'MT079815.1',
    'JF895530.1': 'AB362325.1',
    'HM535007.1': 'LC596057.1',
    'KP330620.1': 'KP330657.1',
    'NC_067066.1': 'OR549572.1',
    'MN103354.1': 'MN103356.1',
    'KJ654246.1': 'KJ654284.1',
    'KJ595343.1': 'KX228124.1',
}

for missing, source in identical_seqs.items():
    iqtree_dist[missing] = iqtree_dist[source]
    iqtree_dist.loc[missing] = iqtree_dist.loc[source]
    iqtree_dist.at[missing, missing] = 0.0

print(f"Distance matrix shape after restoring identical sequences: {iqtree_dist.shape}")

In [4]:
tsne = TSNE(metric='precomputed', perplexity=30, init='random', random_state=42)
coords = tsne.fit_transform(iqtree_dist.values)
coords

array([[  9.911906 ,   0.5183033],
       [ -8.538894 ,  26.728495 ],
       [ -8.508081 ,  26.709314 ],
       ...,
       [  1.6187717, -41.46011  ],
       [ -5.20252  , -36.043804 ],
       [-20.476204 , -24.708063 ]], shape=(1315, 2), dtype=float32)

In [5]:
tax_df = pd.read_csv('data/cox2_th_final/output/cox2_th-meta-clean.tsv', sep='\t', index_col='accession')
tax_df = tax_df.loc[iqtree_dist.columns]

In [20]:
plot(coords, tax_df, 'order', colormap_path='colormap-oom-order.csv', output='tsne-cox2-th-oom')

/tmp/ipykernel_384545/3570168936.py:61: DeprecationWarning:


Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).


